In [1]:
import re
import numpy as np
import pandas as pd
import sqlite3
import xgboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
con = sqlite3.connect('../Dengue20X_timeseries_CPA_NoiseReduction.db')

In [3]:
#Load harmonized data
harmony = pd.read_sql_query('SELECT * from harmony_img_level', con)
harmony.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V15,V16,V17,V18,V19,V20,Image_Metadata_WellID,Image_Metadata_PlateID,ImageNumber,con
0,0.305695,-0.510150,-0.167764,-0.367259,-0.038895,0.081975,0.029306,-0.179630,-0.086734,0.073944,...,-0.055696,0.028117,0.020501,0.115348,0.082253,-0.099590,A01,TimeSeries_20221028_164132,1.0,0
1,0.308491,-0.534324,-0.078357,-0.347420,0.015375,0.167542,0.065436,-0.146419,-0.066748,0.041076,...,-0.050452,-0.016776,0.002821,0.105768,0.069213,-0.050671,A01,TimeSeries_20221028_164132,2.0,0
2,0.229197,-0.630664,-0.010510,-0.350601,-0.134096,0.000786,0.196617,-0.031949,-0.030125,-0.136287,...,-0.038927,-0.039317,0.040172,0.120463,0.117115,0.027497,A01,TimeSeries_20221028_164132,3.0,0
3,0.219075,-0.603700,-0.136947,-0.363700,-0.188136,-0.085320,0.145746,-0.135266,-0.083496,-0.080876,...,-0.040474,-0.016773,0.027725,0.132825,0.081961,0.036667,A01,TimeSeries_20221028_164132,4.0,0
4,0.180284,-0.633803,-0.181292,-0.433847,-0.078915,0.095016,-0.024631,-0.103691,-0.116034,-0.081669,...,-0.013972,0.004544,0.093749,0.081393,0.077621,-0.061731,A01,TimeSeries_20221028_164132,5.0,0


In [12]:
#Subset testing data
DR_only = harmony[harmony['Image_Metadata_PlateID'] == "DR_20221209_143805"]
DR_subset = DR_only[-(DR_only['con'] == '0')]
DR_subset = DR_subset[-(DR_subset['con'] == '48')]
DR_subset.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V15,V16,V17,V18,V19,V20,Image_Metadata_WellID,Image_Metadata_PlateID,ImageNumber,con
1941,0.415536,-0.295184,-0.045193,-0.132086,-0.133866,0.124180,0.011438,-0.113830,-0.057967,0.065978,...,0.040034,-0.097510,0.071565,0.089856,-0.200934,-0.000479,A03,DR_20221209_143805,19.0,none
1942,0.456592,-0.298098,-0.056029,-0.133782,-0.132597,0.110543,0.093774,-0.123562,-0.046664,0.132358,...,0.039597,-0.087993,0.086857,0.014012,-0.109631,0.061197,A03,DR_20221209_143805,20.0,none
1943,0.362407,-0.439106,0.031911,-0.267383,-0.136593,0.108142,0.036040,-0.107179,-0.006272,0.068159,...,0.024527,-0.049610,0.099805,0.039869,-0.132584,0.096879,A03,DR_20221209_143805,21.0,none
1944,0.082208,-0.524462,0.351696,0.115399,-0.227861,0.175707,0.149529,-0.019378,-0.037491,0.032306,...,0.137128,-0.069406,-0.062962,-0.000974,-0.030429,0.027134,A03,DR_20221209_143805,22.0,none
1945,0.260012,-0.329929,0.047248,-0.291195,-0.128030,0.118846,0.096645,-0.114615,-0.042116,0.053652,...,0.070239,-0.043388,0.074300,-0.017036,-0.079012,0.046513,A03,DR_20221209_143805,23.0,none


In [15]:
harmony_drop = harmony.drop(DR_subset.index)

In [16]:
#select control wells
#loc by plate ID and well ID for their respective controls
TS_0 = harmony_drop.loc[(harmony_drop['con'] == "0") & (harmony_drop['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]
TS_12 = harmony_drop.loc[(harmony_drop['con'] == "12") & (harmony_drop['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]
TS_20 = harmony_drop.loc[(harmony_drop['con'] == "20") & (harmony_drop['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]
TS_28 = harmony_drop.loc[(harmony_drop['con'] == "28") & (harmony_drop['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]
TS_36 = harmony_drop.loc[(harmony_drop['con'] == "36") & (harmony_drop['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]
TS_48 = harmony_drop.loc[(harmony_drop['con'] == "48") & (harmony_drop['Image_Metadata_PlateID'] == 'TimeSeries_20221028_164132')]

DR_0 = harmony_drop.loc[(harmony_drop['con'] == "0") & (harmony_drop['Image_Metadata_PlateID'] == 'DR_20221209_143805')]
DR_48 = harmony_drop.loc[(harmony_drop['con'] == "48") & (harmony_drop['Image_Metadata_PlateID'] == 'DR_20221209_143805')]


In [35]:
meta_cols = harmony_labeled.columns[harmony_labeled.columns.str.contains(pat='Metadata|ImageNumber|con', flags=re.IGNORECASE)].tolist()


NameError: name 'harmony_labeled' is not defined

In [15]:
cols = harmony_labeled.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()

In [16]:
X = harmony_labeled[cols]
y = harmony_labeled['label']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [19]:
model = xgboost.XGBRegressor()

In [20]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             n_estimators=100, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=None, ...)

In [21]:
preds = model.predict(X_test)

In [22]:
r2 = r2_score(y_true=y_test, y_pred=preds)
mse = mean_squared_error(y_true=y_test, y_pred=preds)

In [23]:
print("R2", r2)
print("MSE", mse)

R2 0.9441372688811074
MSE 0.013924363468265491


In [20]:
times = ["12", "20", "28", "36", "48"]
TS_0['label'] = 0
DR_0['label'] = 0
model = xgboost.XGBRegressor()

for time in times:
    training = pd.DataFrame()
    if time == "12":
        TS_12['label'] = 1
        training = pd.concat([TS_0, DR_0, TS_12], ignore_index = True)
    elif time == "20":
        TS_20['label'] = 1
        training = pd.concat([TS_0, DR_0, TS_20], ignore_index = True)
    elif time == '28':
        TS_28['label'] = 1
        training = pd.concat([TS_0, DR_0, TS_28], ignore_index = True)
    elif time == '36':
        TS_36['label'] = 1
        training = pd.concat([TS_0, DR_0, TS_36], ignore_index = True)
    elif time == '48':
        TS_48['label'] = 1
        DR_48['label'] = 1
        training = pd.concat([TS_0, DR_0, TS_48, DR_48], ignore_index = True)
        
    meta_cols = training.columns[training.columns.str.contains(pat='Metadata|ImageNumber|con', flags=re.IGNORECASE)].tolist()
    cols = training.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()
    
    X = training[cols]
    y = training['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_true=y_test, y_pred=preds)
    mse = mean_squared_error(y_true=y_test, y_pred=preds)
    print(time)
    print("R2", r2)
    print("MSE", mse)
    print()
    
    model.save_model('xgb_model_0_' + time + '_scoreDR')
    
    scores = model.predict(DR_subset[cols])
    DR_subset['score'] = scores
    filter_DR = DR_subset[["Image_Metadata_WellID", "Image_Metadata_PlateID", "ImageNumber", "score"]]
    filter_DR.to_csv("DR_scored_0_" + time + ".csv")
    

<ipython-input-20-ba32d5b74fd9>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_0['label'] = 0
<ipython-input-20-ba32d5b74fd9>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DR_0['label'] = 0
<ipython-input-20-ba32d5b74fd9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versu

12
R2 0.7409681035516419
MSE 0.0622151829266056



<ipython-input-20-ba32d5b74fd9>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_20['label'] = 1


20
R2 0.7481708021382991
MSE 0.049887945152533274

28
R2 0.9360908880487454
MSE 0.012428882651902068



<ipython-input-20-ba32d5b74fd9>:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_28['label'] = 1
<ipython-input-20-ba32d5b74fd9>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_36['label'] = 1


36
R2 0.9277522820948265
MSE 0.013490179956831898



<ipython-input-20-ba32d5b74fd9>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_48['label'] = 1
<ipython-input-20-ba32d5b74fd9>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DR_48['label'] = 1


48
R2 0.8913122430455056
MSE 0.02715178109811137



In [18]:
times = ["12", "20", "28", "36", "48"]

for i in range(len(times)):
    for j in range(i + 1, len(times)):
        training = pd.DataFrame()
        t1,t2 = times[i], times[j]
        if t1 == "12" and t2 == "20":
            TS_12['label'] = 0
            TS_20['label'] = 1
            training = pd.concat([TS_12, TS_20], ignore_index = True)
        elif t1 == "12" and t2 == "28":
            TS_12['label'] = 0
            TS_28['label'] = 1
            training = pd.concat([TS_12, TS_28], ignore_index = True)
        elif t1 == "12" and t2 == "36":
            TS_12['label'] = 0
            TS_36['label'] = 1
            training = pd.concat([TS_12, TS_36], ignore_index = True)  
        elif t1 == "12" and t2 == "48":
            TS_12['label'] = 0
            TS_48['label'] = 1
            DR_48['label'] = 1
            training = pd.concat([TS_12, TS_48, DR_48], ignore_index = True)
        elif t1 == "20" and t2 == "28":
            TS_20['label'] = 0
            TS_28['label'] = 1
            training = pd.concat([TS_20, TS_28], ignore_index = True)
        elif t1 == "20" and t2 == "36":
            TS_20['label'] = 0
            TS_36['label'] = 1
            training = pd.concat([TS_20, TS_36], ignore_index = True)
        elif t1 == "20" and t2 == "48":
            TS_20['label'] = 0
            TS_48['label'] = 1
            DR_48['label'] = 1
            training = pd.concat([TS_20, TS_48, DR_48], ignore_index = True)
        elif t1 == "28" and t2 == "36":
            TS_28['label'] = 0
            TS_36['label'] = 1
            training = pd.concat([TS_28, TS_36], ignore_index = True)
        elif t1 == "28" and t2 == "48":
            TS_28['label'] = 0
            TS_48['label'] = 1
            DR_48['label'] = 1
            training = pd.concat([TS_28, TS_48, DR_48], ignore_index = True)
        elif t1 == "36" and t2 == "48":
            TS_36['label'] = 0
            TS_48['label'] = 1
            DR_48['label'] = 1
            training = pd.concat([TS_36, TS_48, DR_48], ignore_index = True)
            
        meta_cols = training.columns[training.columns.str.contains(pat='Metadata|ImageNumber|con', flags=re.IGNORECASE)].tolist()
        cols = training.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()
    
        X = training[cols]
        y = training['label']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        r2 = r2_score(y_true=y_test, y_pred=preds)
        mse = mean_squared_error(y_true=y_test, y_pred=preds)
        print(t1 + " " + t2)
        print("R2", r2)
        print("MSE", mse)
        print()
    
        model.save_model('xgb_model_' + t1 + '_' + t2 + '_scoreDR')
    
        scores = model.predict(DR_subset[cols])
        DR_subset['score'] = scores
        filter_DR = DR_subset[["Image_Metadata_WellID", "Image_Metadata_PlateID", "ImageNumber", "score"]]
        filter_DR.to_csv("DR_scored_" + t1 + "_" + t2 + ".csv")
            

<ipython-input-18-61a546adcc7c>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_12['label'] = 0
<ipython-input-18-61a546adcc7c>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_20['label'] = 1


12 20
R2 0.6539143350784602
MSE 0.07516332231230248

12 28
R2

<ipython-input-18-61a546adcc7c>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_12['label'] = 0
<ipython-input-18-61a546adcc7c>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_28['label'] = 1
<ipython-input-18-61a546adcc7c>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

 0.9411138272164326
MSE 0.012382539144669169

12 36
R2 0.9418188450591973
MSE 0.010565560030374161



<ipython-input-18-61a546adcc7c>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_12['label'] = 0
<ipython-input-18-61a546adcc7c>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_48['label'] = 1
<ipython-input-18-61a546adcc7c>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

...

20 36
R2 0.8196030956156637
MSE 0.043552613404174476

20 48
R2 0.8815660168020776
MSE 0.022696980999172618



<ipython-input-18-61a546adcc7c>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_28['label'] = 0
<ipython-input-18-61a546adcc7c>:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_36['label'] = 1
<ipython-input-18-61a546adcc7c>:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

28 36
R2 0.5160157116557791
MSE 0.11445220664062229

28 48
R2 0.7776873196103533
MSE 0.04358759770188479



<ipython-input-18-61a546adcc7c>:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_36['label'] = 0
<ipython-input-18-61a546adcc7c>:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TS_48['label'] = 1
<ipython-input-18-61a546adcc7c>:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

36 48
R2 0.6017450326597402
MSE 0.06372079477444156



In [22]:
times = ["12", "20", "28", "36", "48"]

for i in range(len(times)):
    for j in range(i + 1, len(times)):
        t1,t2 = times[i], times[j]
        model.load_model('xgb_model_' + t1 + '_' + t2 + '_scoreDR')
        
        scores = model.predict(DR_subset[cols])
        DR_subset['score'] = scores
        filter_DR = DR_subset[["Image_Metadata_WellID", "Image_Metadata_PlateID", "ImageNumber", "score"]]
        filter_DR.to_csv("DR_scored_" + t1 + "_" + t2 + ".csv")

In [25]:
scores = model.predict(DR_subset[cols])

In [26]:
DR_subset['score'] = scores

In [27]:
DR_subset = DR_subset[["Image_Metadata_WellID", "Image_Metadata_PlateID", "ImageNumber", "score"]]
DR_subset.head()

,Image_Metadata_WellID,Image_Metadata_PlateID,ImageNumber,score
2290,B17,DR_20221209_143805,368,1.000143
4683,O12,DR_20221209_143805,2914,1.000264
3253,G05,DR_20221209_143805,1340,0.999975
4059,J23,DR_20221209_143805,2146,0.003817
2444,C10,DR_20221209_143805,522,0.998431


In [28]:
DR_subset.to_csv("DR_scored_0_48.csv")